In [2]:
using DynamicPolynomials

In [8]:
function lindblad_rhs(ρ, H, J::Array)
    """
    Right hand side of the Lindblad master equation with multiple dicipators
    """
   
    Σ = sum([ ( Jⱼ * ρ * Jⱼ' - (Jⱼ' * Jⱼ  * ρ + ρ * Jⱼ' * Jⱼ)/2 ) for Jⱼ in J ])
    
    return -im * (H * ρ - ρ * H) + Σ 
    
end


function lindblad_rhs(ρ, H, J::Array, g)
    """
    Right hand side of the Lindblad master equation with multiple dicipators
    """
   
    Σ = sum([ ( Jⱼ * ρ * Jⱼ' - (Jⱼ' * Jⱼ  * ρ + ρ * Jⱼ' * Jⱼ)/2 ) for Jⱼ in J ])
    
    return -im * (H * ρ - ρ * H) + g * Σ 

end

    

lindblad_rhs (generic function with 2 methods)

In [9]:
function frobenius_norm2(m)
    return tr(m * m')
end

frobenius_norm2 (generic function with 1 method)

The most general (time-independent) master equation has the following form 
\begin{align}
    \dot\rho = \mathcal{L}\rho(t) + \int_{t-\tau}^t \mathcal{M}(s) \rho(s) ds
\end{align} 
We can discretize this 
\begin{align}
    \dot\rho & = \mathcal{L}\rho(t) + \sum_{n=0}^{N-1} \mathcal{M}(n\Delta t - \tau) \rho(n\Delta t - \tau) \Delta t  \\
        & = \mathcal{L}\rho(t) + \sum_{n=0}^{N-1} \mathcal{M}_n \rho(n\Delta t - \tau)  
\end{align}
Now each of the $\mathcal{M}_n$ must be a completely positive map, so they must have the same structure as $\mathcal{L}$, which means they have the same form as the GKSL master equation. So we can write the non-Markovian master equation as 
\begin{align}
    \dot\rho & = \sum_{j=0}^{N} \mathcal{L}_j \rho(t - j\Delta t) 
\end{align}
with $\Delta t = \tau/N$. 

For the non-Markovian objective with memory k steps back

$$r_i = \rho[i] - \rho[i-2] - \Delta t \mathcal{L}_0\left(\tfrac{\rho[i-2] + 4\rho[i-1] + \rho[i]}{3}\right) - \Delta t \mathcal{M}\left(\tfrac{\rho[i-k-2] + 4\rho[i-k-1] + \rho[i-k]}{3}\right)$$

$\mathcal{L}_0$ = standard Lindblad at current time (uses H, J)

$\mathcal{M}$ = memory kernel at time $t - k\Delta t$ — same Lindblad form but needs its own operators

In [10]:
function non_mark_obj(ρ::Array{ComplexF64,3}, t, H, J, H_mem, J_mem, k::Int)
    obj = 0
    for i in (k+3):size(ρ, 3)
        ρ_avg  = (ρ[:,:,i-2]   + 4ρ[:,:,i-1]   + ρ[:,:,i])   / 3
        ρ_mem  = (ρ[:,:,i-k-2] + 4ρ[:,:,i-k-1] + ρ[:,:,i-k]) / 3
        
        residual = ρ[:,:,i] - ρ[:,:,i-2] -
                   (t[i]-t[i-1]) * lindblad_rhs(ρ_avg, H, J) -
                   (t[i]-t[i-1]) * lindblad_rhs(ρ_mem, H_mem, J_mem)
        
        obj += frobenius_norm2(residual)
    end
    obj = sum(real(coef) * mon for (coef, mon) in zip(coefficients(obj), monomials(obj)))
    return obj
end


non_mark_obj (generic function with 2 methods)

In [12]:
# Define polynomial variables
@polyvar w₁ r₁ wm rm

H₁ˢʸᵐᵇ = [ w₁   0
          0   0. ]

J₁ˢʸᵐᵇ = [ 0   r₁
           0   0. ]


Hmˢʸᵐᵇ = [ wm   0
          0   0. ]

Jmˢʸᵐᵇ = [ 0   rm
           0   0. ]        


2×2 Matrix{Term{true, Float64}}:
 0.0  rm
 0.0  0.0